# ComfyUI Fixed Colab

Run cells from top to bottom. This fixed notebook installs or updates ComfyUI on Google Drive, installs ComfyUI-Manager, fixes dependency issues like `alembic`, removes stale files that can cause `comfy_aimdo` import errors, downloads SD1.5 + VAE, and launches ComfyUI through Cloudflare Tunnel.

## 1. Check GPU

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=False)


## 2. Install or update ComfyUI

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import shutil
import subprocess

USE_GOOGLE_DRIVE = True
CLEAN_STALE_FILES = True
WORKSPACE = Path('/content/drive/MyDrive/ComfyUI') if USE_GOOGLE_DRIVE else Path('/content/ComfyUI')
UPSTREAM = 'https://github.com/comfyanonymous/ComfyUI.git'


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


def merge_old_models(backup, workspace):
    old_models = backup / 'models'
    new_models = workspace / 'models'
    if not old_models.exists():
        return
    new_models.mkdir(parents=True, exist_ok=True)
    for old_child in old_models.iterdir():
        target_dir = new_models / old_child.name
        if old_child.is_dir():
            target_dir.mkdir(parents=True, exist_ok=True)
            for item in old_child.iterdir():
                target = target_dir / item.name
                if target.exists():
                    continue
                shutil.move(str(item), str(target))
                print('Restored model file:', target)
        else:
            target = new_models / old_child.name
            if not target.exists():
                shutil.move(str(old_child), str(target))
                print('Restored model file:', target)


def fresh_clone_with_backup(workspace):
    backup = None
    if workspace.exists():
        backup = workspace.with_name('ComfyUI_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
        print('Existing ComfyUI checkout is not updateable. Moving it to:', backup)
        shutil.move(str(workspace), str(backup))
    run(['git', 'clone', '--depth', '1', UPSTREAM, str(workspace)])
    if backup is not None:
        merge_old_models(backup, workspace)


if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')

WORKSPACE.parent.mkdir(parents=True, exist_ok=True)

if not WORKSPACE.exists():
    run(['git', 'clone', '--depth', '1', UPSTREAM, str(WORKSPACE)])
elif not (WORKSPACE / '.git').exists():
    fresh_clone_with_backup(WORKSPACE)

if CLEAN_STALE_FILES and WORKSPACE.exists():
    stale_paths = [
        WORKSPACE / 'comfy_aimdo',
        WORKSPACE / 'comfy_aimdo.py',
        WORKSPACE / 'comfy_api_nodes',
    ]
    for path in stale_paths:
        if path.is_dir():
            shutil.rmtree(path)
            print('Removed stale directory:', path)
        elif path.exists():
            path.unlink()
            print('Removed stale file:', path)

# Try to update the existing checkout. If Git/Drive corruption blocks fetch, rebuild cleanly.
run(['git', 'remote', 'set-url', 'origin', UPSTREAM], cwd=WORKSPACE)
fetch_result = run(['git', 'fetch', '--depth', '1', 'origin'], cwd=WORKSPACE, check=False)
if fetch_result.returncode != 0:
    fresh_clone_with_backup(WORKSPACE)
else:
    branch_result = subprocess.run(['git', 'rev-parse', '--verify', 'origin/master'], cwd=str(WORKSPACE), check=False)
    upstream_branch = 'origin/master' if branch_result.returncode == 0 else 'origin/main'
    run(['git', 'reset', '--hard', upstream_branch], cwd=WORKSPACE)

run(['python3', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run(['python3', '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121'])
run(['python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=WORKSPACE)
run([
    'python3', '-m', 'pip', 'install', '-q',
    'alembic', 'blake3', 'GitPython', 'accelerate', 'einops', 'transformers',
    'safetensors', 'aiohttp', 'pyyaml', 'Pillow', 'scipy', 'tqdm', 'psutil',
    'tokenizers', 'torchsde', 'kornia', 'spandrel', 'soundfile', 'sentencepiece'
])

print('ComfyUI setup complete:', WORKSPACE)


## 3. Install or update ComfyUI-Manager

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
custom_nodes = WORKSPACE / 'custom_nodes'
manager = custom_nodes / 'ComfyUI-Manager'
custom_nodes.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

if not manager.exists():
    run(['git', 'clone', 'https://github.com/ltdrdata/ComfyUI-Manager.git', str(manager)])

run(['git', 'reset', '--hard', 'HEAD'], cwd=manager)
run(['git', 'pull', '--ff-only'], cwd=manager)
run(['python3', 'custom_nodes/ComfyUI-Manager/cm-cli.py', 'restore-dependencies'], cwd=WORKSPACE)

print('ComfyUI-Manager setup complete')


## 4. Download starter models

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
(WORKSPACE / 'models' / 'checkpoints').mkdir(parents=True, exist_ok=True)
(WORKSPACE / 'models' / 'vae').mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

run([
    'wget', '-c',
    'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt',
    '-P', str(WORKSPACE / 'models' / 'checkpoints')
])
run([
    'wget', '-c',
    'https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors',
    '-P', str(WORKSPACE / 'models' / 'vae')
])


## 5. Launch with dual public links

Starts ComfyUI, waits only for the lightweight `/system_stats` health check, then creates both Cloudflare and LocalTunnel links. If one link is slow or blank, use the other.


In [ ]:
from pathlib import Path
from google.colab import output
import queue
import re
import socket
import subprocess
import threading
import time
import urllib.request

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
PORT = 8188


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


def wait_for_port(port, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            return
        time.sleep(0.5)
    raise TimeoutError(f'Port {port} did not open in {timeout}s')


def fetch_url(url, timeout=12):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        response.read(512)
        return response.status


def wait_for_system_stats(port, timeout=240):
    print('Waiting for ComfyUI backend /system_stats...')
    wait_for_port(port, timeout=timeout)
    start = time.time()
    last_error = None
    while time.time() - start < timeout:
        try:
            if fetch_url(f'http://127.0.0.1:{port}/system_stats') == 200:
                print('ComfyUI backend is ready.')
                return
        except Exception as exc:
            last_error = exc
        time.sleep(1)
    raise TimeoutError(f'ComfyUI backend did not become ready. Last error: {last_error}')


def quick_probe(name, url):
    for attempt in range(8):
        try:
            status = fetch_url(url + '/system_stats', timeout=8)
            if status == 200:
                print(f'{name} probe OK:', url)
                return True
        except Exception as exc:
            print(f'{name} probe retry {attempt + 1}/8:', exc)
            time.sleep(2)
    print(f'{name} probe did not pass yet. The link may still work after DNS catches up:', url)
    return False


def start_cloudflared(port, out_q):
    try:
        proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )
        start = time.time()
        while time.time() - start < 120:
            line = proc.stderr.readline()
            if not line:
                time.sleep(0.2)
                continue
            match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                out_q.put(('Cloudflare', url, proc))
                return
        out_q.put(('Cloudflare', None, proc))
    except Exception as exc:
        out_q.put(('Cloudflare error', str(exc), None))


def start_localtunnel(port, out_q):
    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=20).read().decode().strip()
        print('LocalTunnel password / endpoint IP:', ip)
        proc = subprocess.Popen(
            ['npx', '--yes', 'localtunnel', '--port', str(port)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        start = time.time()
        while time.time() - start < 120:
            line = proc.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            print('localtunnel:', line.rstrip())
            match = re.search(r'https://[-a-zA-Z0-9.]+\.loca\.lt', line)
            if match:
                url = match.group(0)
                out_q.put(('LocalTunnel', url, proc))
                return
        out_q.put(('LocalTunnel', None, proc))
    except Exception as exc:
        out_q.put(('LocalTunnel error', str(exc), None))


run(['wget', '-q', '-O', '/content/cloudflared-linux-amd64.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared-linux-amd64.deb'], check=False)

print('Starting ComfyUI server...')
comfy_proc = subprocess.Popen(
    ['python3', 'main.py', '--dont-print-server'],
    cwd=str(WORKSPACE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

wait_for_system_stats(PORT)
print('Showing ComfyUI inside this Colab page...')
output.serve_kernel_port_as_iframe(PORT, height=900)

links = queue.Queue()
threading.Thread(target=start_cloudflared, daemon=True, args=(PORT, links)).start()
threading.Thread(target=start_localtunnel, daemon=True, args=(PORT, links)).start()

found = {}
deadline = time.time() + 150
while time.time() < deadline and len(found) < 2:
    try:
        name, url, proc = links.get(timeout=5)
    except queue.Empty:
        continue
    if url and url.startswith('http'):
        found[name] = url
        print(f'{name} public URL:', url)
        quick_probe(name, url)
    else:
        print(name, 'did not provide a usable URL:', url)

print()
print('READY LINKS')
if 'Cloudflare' in found:
    print('Cloudflare:', found['Cloudflare'])
if 'LocalTunnel' in found:
    print('LocalTunnel:', found['LocalTunnel'])
    print('LocalTunnel password / endpoint IP is printed above. Enter it if the page asks for a password.')
if not found:
    print('No public tunnel URL was created. Re-run this cell once.')
print('Use whichever link opens faster on your phone. If one is blank, try the other.')
print('Keep this cell running. Stop it only when you want to shut down ComfyUI.')

for line in comfy_proc.stdout:
    print(line, end='')


## 5B. Regenerate public links only

Use this only if step 5 started ComfyUI but did not print public links. It does not reinstall or restart ComfyUI; it only creates new Cloudflare and LocalTunnel links for the already-running server on port 8188.


In [ ]:
import queue
import re
import socket
import subprocess
import threading
import time
import urllib.request

PORT = 8188


def run(cmd, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, check=check)


def port_is_open(port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', port))
    sock.close()
    return result == 0


def fetch_url(url, timeout=8):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        response.read(256)
        return response.status


def start_cloudflared(port, out_q):
    try:
        run(['wget', '-q', '-O', '/content/cloudflared-linux-amd64.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=False)
        run(['dpkg', '-i', '/content/cloudflared-linux-amd64.deb'], check=False)
        proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )
        start = time.time()
        while time.time() - start < 120:
            line = proc.stderr.readline()
            if not line:
                time.sleep(0.2)
                continue
            match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
            if match:
                out_q.put(('Cloudflare', match.group(0)))
                return
        out_q.put(('Cloudflare', None))
    except Exception as exc:
        out_q.put(('Cloudflare error', str(exc)))


def start_localtunnel(port, out_q):
    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=20).read().decode().strip()
        print('LocalTunnel password / endpoint IP:', ip)
        proc = subprocess.Popen(
            ['npx', '--yes', 'localtunnel', '--port', str(port)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        start = time.time()
        while time.time() - start < 120:
            line = proc.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            print('localtunnel:', line.rstrip())
            match = re.search(r'https://[-a-zA-Z0-9.]+\.loca\.lt', line)
            if match:
                out_q.put(('LocalTunnel', match.group(0)))
                return
        out_q.put(('LocalTunnel', None))
    except Exception as exc:
        out_q.put(('LocalTunnel error', str(exc)))


if not port_is_open(PORT):
    print('ComfyUI is not running on port 8188. Run step 5 first, or rerun step 2 then step 5.')
else:
    try:
        status = fetch_url(f'http://127.0.0.1:{PORT}/system_stats')
        print('ComfyUI backend is reachable:', status)
    except Exception as exc:
        print('Port is open, but /system_stats failed:', exc)

    links = queue.Queue()
    threading.Thread(target=start_cloudflared, daemon=True, args=(PORT, links)).start()
    threading.Thread(target=start_localtunnel, daemon=True, args=(PORT, links)).start()

    found = {}
    deadline = time.time() + 150
    while time.time() < deadline and len(found) < 2:
        try:
            name, url = links.get(timeout=5)
        except queue.Empty:
            continue
        if url and str(url).startswith('http'):
            found[name] = url
            print(f'{name}:', url)
        else:
            print(name, 'failed:', url)

    print()
    print('READY LINKS')
    if 'Cloudflare' in found:
        print('Cloudflare:', found['Cloudflare'])
    if 'LocalTunnel' in found:
        print('LocalTunnel:', found['LocalTunnel'])
        print('If LocalTunnel asks for password, use the endpoint IP printed above.')
    if not found:
        print('No link was created. Re-run this cell once, or run step 5 again.')
